# Skills as compiled artifacts

In this notebook, we compile a skill from reusable instruction fragments. For each combination of task variables, the compiler assembles the applicable fragments into a set of instructions the model can use directly. The harness then validates the output against a schema and uses judges to check each part against the rules that apply to it. If a part fails that check, we can repair it separately.

| Step | What we build | How we use it |
|---|---|---|
| 1 | **Fragments.** Small files that own named sections and declare when they apply. | We edit the instructions here. |
| 2 | **Compiled leaves.** One flat set of instructions per variant, with the conditions already resolved. | The generator receives the leaf for the current task. |
| 3 | **Section index and manifest.** A way to look up individual sections and trace them back to their source fragments. | The harness retrieves the rules that apply to each part. |
| 4 | **Typed envelope.** An output schema derived from the skill’s contract section. | The harness validates the output structure and field types. |
| 5 | **Per-part judging and repair.** Independent judges evaluate each part using only its relevant rules. | The harness can repair the part that failed the semantic check. |

The example generates release notes using two task variables. `tone` can be `formal` or `casual`, and `audience` can be `customer` or `developer`. We maintain five instruction fragments and compile them into four variants.

We load the fragments from Google Drive, generate candidates with a live model, and evaluate them with three judges from different model families through OpenRouter. You’ll need an `OPENROUTER_API_KEY` to run the live example. A complete run costs a few cents. Section 21 shows how we calculate that cost from measured token usage and published per-token rates.

Section 20 uses stub judges to test the harness logic in CI. The stubs return fixed responses, so we can regression-test the harness without API credentials, inference costs, or variation in the judges’ responses.

In [ ]:
!pip install instructor

In [ ]:
# gdown 6 dropped the `remaining_ok` argument that gdown 5 needs for
# partial folder listings, so the call below is built by signature inspection
# rather than pinned to one release.
try:
    import gdown
except ModuleNotFoundError:
    %pip install -q gdown
    import gdown

import asyncio, hashlib, inspect, json, os, re, shutil, statistics, textwrap, time
from dataclasses import dataclass, field
from functools import lru_cache
from pathlib import Path
from typing import Awaitable, Callable, Dict, List, Mapping, Optional, Sequence, Set, Tuple

import yaml
from pydantic import BaseModel, Field, ValidationError, field_validator

DRIVE_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1B3RvSOAtf-7ccdJtlRZ8FuBIVplX40_-?usp=sharing"
)

WORK = Path("skill_demo").resolve()
SRC, GENERATED = WORK / "skills_src", WORK / "generated"

# Order matters: clear the tree FIRST, then download into it. Doing it the
# other way round deletes what was just fetched.
shutil.rmtree(WORK, ignore_errors=True)
SRC.mkdir(parents=True); GENERATED.mkdir(parents=True)

kwargs = dict(url=DRIVE_FOLDER_URL, output=str(SRC), quiet=True, use_cookies=False)
if "remaining_ok" in inspect.signature(gdown.download_folder).parameters:
    kwargs["remaining_ok"] = True
downloaded = gdown.download_folder(**kwargs)

if not downloaded:
    raise RuntimeError(
        "No files downloaded. The Drive folder must be shared with "
        "'Anyone with the link'."
    )
print(f"working in {WORK}")
print(f"downloaded {len(downloaded)} fragments:")
for p in sorted(Path(f).name for f in downloaded):
    print("   ", p)

working in /content/skill_demo
downloaded 5 fragments:
    audience-customer.md
    audience-developer.md
    base-release-notes.md
    tone-casual.md
    tone-formal.md


## 1. Defining an instruction fragment

A fragment is a small Markdown file with YAML frontmatter. The frontmatter tells the compiler when to use the fragment and what to do with its sections.

- `applies_when` matches task variables such as `tone: casual`. If we leave it out, the fragment applies to every task. That’s how the base fragment works.
- `replaces` and `appends` name the sections the fragment replaces or adds content to.
- `must_replace` is only used in the base fragment. It names placeholder sections that a matching overlay must replace.

In the Markdown body, we mark each section with `<!-- section: id -->`. The harness, validator, and judges use the same IDs to refer to specific sections.

For example, suppose the base fragment contains formal-tone guidance and the casual overlay forgets to replace it. Both files can look fine on their own, but the compiled casual variant will still tell the model to write formally. If we list that section in `must_replace`, the compiler catches the missing replacement during assembly.

In [ ]:
print((SRC / "base-release-notes.md").read_text()[:1200])
print("...\n" + "=" * 70 + "\n")
print((SRC / "tone-casual.md").read_text()[:600])

---
fragment_id: base-release-notes
kind: base
version: 1.0.0
applies_when: {}
provides: [output.contract, summary.style, highlights.structure, address.contract, prohibitions.hard, typography]
must_replace: [highlights.structure, address.contract, prohibitions.hard]
---

<!-- section: output.contract -->
## 1. Output contract

Return one JSON object with exactly these keys:

- `summary`: one string.
- `highlights`: a list of strings.

No prose outside the JSON object. No key that is not listed here.

<!-- section: summary.style -->
## 2. Summary

One sentence, at most 25 words. It names what changed, not why it is good.

<!-- section: highlights.structure -->
## 3. Highlights

PLACEHOLDER - the tone fragment owns this section.

<!-- section: address.contract -->
## 4. Addressing the reader

PLACEHOLDER - the tone fragment owns this section.

<!-- section: prohibitions.hard -->
## 5. Prohibitions

PLACEHOLDER - the tone fragment owns this section.

<!-- section: typography -->
## 6. Typ

## 2. Validating fragments before assembly

Before we combine the sections, the parser checks three things.

- **Section declarations match the body.** Every section named in `replaces` or `appends` must exist in the fragment’s body, and every section in the body must be declared. This keeps the manifest’s ownership records in sync with the instructions we compile.
- **Each fragment owns only permitted sections.** `PERMITTED` defines which sections each fragment type can own. A `tone` fragment can’t replace the output contract. An `audience` fragment can own sections matching `summary.*` and the specific section `output.contract`. Giving it access to another `output.*` section requires an explicit change to `PERMITTED`.
- **`applies_when` uses declared axes.** A typo like `audiance: developer` would keep the fragment out of every compiled variant. The parser rejects the unknown axis so we can fix the typo before assembly.

In [ ]:
_FRONTMATTER = re.compile(r"\A---\n(.*?)\n---\n?", re.S)
_SECTION_MARK = re.compile(r"^<!--\s*section:\s*([A-Za-z0-9._-]+)\s*-->\s*$", re.M)
_PARAM = re.compile(r"\{\{\s*([A-Za-z0-9_]+)\s*\}\}")

PERMITTED: Dict[str, Sequence[str]] = {
    "base":     ("*",),
    "tone":     ("address.*", "highlights.*", "prohibitions.*"),
    "audience": ("summary.*", "output.contract"),
}
AXIS_ORDER = ("tone", "audience")


class CompileError(Exception):
    '''Assembly failed. Never retried: a bad assembly is a harness defect.'''


@dataclass
class Fragment:
    fragment_id: str
    kind: str
    version: str
    applies_when: Dict[str, object] = field(default_factory=dict)
    provides: List[str] = field(default_factory=list)
    must_replace: List[str] = field(default_factory=list)
    replaces: List[str] = field(default_factory=list)
    appends: List[str] = field(default_factory=list)
    param_values: Dict[str, str] = field(default_factory=dict)
    sections: Dict[str, str] = field(default_factory=dict)
    path: str = ""
    raw: str = ""

    @property
    def declared(self) -> List[str]:
        return [*self.provides, *self.replaces, *self.appends]

    @property
    def source_sha256(self) -> str:
        return hashlib.sha256(self.raw.encode()).hexdigest()


def _namespace_ok(kind: str, section_id: str) -> bool:
    for pattern in PERMITTED.get(kind, ()):
        if pattern == "*" or pattern == section_id:
            return True
        if pattern.endswith(".*") and section_id.startswith(pattern[:-1]):
            return True
    return False


def parse_fragment(text: str, *, path: str = "") -> Fragment:
    m = _FRONTMATTER.match(text)
    if not m:
        raise CompileError(f"fragment.no-frontmatter: {path or '<inline>'}")
    meta = yaml.safe_load(m.group(1)) or {}
    body = text[m.end():]

    sections, marks = {}, list(_SECTION_MARK.finditer(body))
    for i, mark in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(body)
        sections[mark.group(1)] = body[mark.end():end].strip("\n")

    frag = Fragment(**meta, sections=sections, path=path, raw=text)

    declared, written = set(frag.declared), set(sections)
    if declared - written:
        raise CompileError(f"fragment.declared-not-written: {sorted(declared - written)} in {frag.fragment_id}")
    if written - declared:
        raise CompileError(f"fragment.written-not-declared: {sorted(written - declared)} in {frag.fragment_id}")
    for sid in declared:
        if not _namespace_ok(frag.kind, sid):
            raise CompileError(f"fragment.namespace-violation: kind '{frag.kind}' may not own '{sid}' ({frag.fragment_id})")
    unknown_axes = set(frag.applies_when) - set(AXIS_ORDER)
    if unknown_axes:
        raise CompileError(f"fragment.unknown-axis: {sorted(unknown_axes)} in {frag.fragment_id} (declared axes: {list(AXIS_ORDER)})")
    return frag


def load_fragments(root: Path) -> List[Fragment]:
    return [parse_fragment(p.read_text(encoding="utf-8"), path=p.name)
            for p in sorted(root.rglob("*.md"))]


frags = load_fragments(SRC)
for f in frags:
    print(f"{f.fragment_id:22} kind={f.kind:9} v{f.version}  applies_when={f.applies_when or '{} (always)'}")
    print(f"{'':22} owns {sorted(f.declared)}")

audience-customer      kind=audience  v1.0.0  applies_when={'audience': 'customer'}
                       owns ['summary.style']
audience-developer     kind=audience  v1.0.0  applies_when={'audience': 'developer'}
                       owns ['output.contract', 'summary.style']
base-release-notes     kind=base      v1.0.0  applies_when={} (always)
                       owns ['address.contract', 'highlights.structure', 'output.contract', 'prohibitions.hard', 'summary.style', 'typography']
tone-casual            kind=tone      v1.0.0  applies_when={'tone': 'casual'}
                       owns ['address.contract', 'highlights.structure', 'prohibitions.hard']
tone-formal            kind=tone      v1.0.0  applies_when={'tone': 'formal'}
                       owns ['address.contract', 'highlights.structure', 'prohibitions.hard']


## 3. Validating claims before assembly

In `compile_leaf`, we collect the section claims from all fragments that apply to the selected variant. We check those claims together before we assemble the instructions.

Suppose one fragment appends a rule to a section and another replaces that same section. If we apply the append first, the replacement can erase the rule we just added. Checking all claims up front lets us reject the conflict regardless of file order.

We also check where parameter values come from. If two fragments give the same parameter different values, compilation fails. Parameters that come from task axes are reserved. For example, if the selected variant has `tone: casual`, `{{tone}}` gets its value from that axis and a fragment can’t redefine it.

| Rejection | When it happens |
|---|---|
| `assembly.two-owners` | Two fragments claim a replacement for the same section. |
| `assembly.replace-unknown-section` | A fragment tries to replace a section the base doesn’t define. |
| `assembly.append-and-replace` | The claims include both an append and a replacement for the same section. |
| `assembly.placeholder-not-replaced` | A section listed in `must_replace` has no replacement. |
| `assembly.param-collision` | Two fragments give the same parameter different values. |
| `assembly.param-shadows-axis` | A fragment defines a parameter reserved for a task axis. |
| `assembly.unresolved-param` | A placeholder like `{{param}}` remains after substitution. |

In [ ]:
def matches(frag: Fragment, axes: Mapping[str, str]) -> bool:
    '''A list of axis values is not a convenience: when two values share one rule
    set, a scalar-only axis forces two byte-identical fragments - reintroducing
    exactly the duplication the compiler exists to remove.'''
    for key, want in frag.applies_when.items():
        got = axes.get(key)
        if isinstance(want, (list, tuple, set)):
            if got not in want:
                return False
        elif got != want:
            return False
    return True


@dataclass
class Leaf:
    skill_id: str
    axes: Dict[str, str]
    order: List[str]
    sections: Dict[str, str]
    owners: Dict[str, str]              # final owner (who replaced it last)
    contributors: Dict[str, List[str]]  # everyone whose text is in there
    provenance: List[Dict[str, str]]    # fragment id, version, source hash
    content: str
    content_sha256: str


def compile_leaf(fragments, axes, *, skill_id, required=()) -> Leaf:
    selected = [f for f in fragments if matches(f, axes)]
    bases = [f for f in selected if f.kind == "base"]
    if len(bases) != 1:
        raise CompileError(f"assembly.base-count: expected 1, got {len(bases)}")
    base, overlays = bases[0], [f for f in selected if f.kind != "base"]

    content = dict(base.sections)
    owner = {sid: base.fragment_id for sid in base.sections}
    contributors = {sid: [base.fragment_id] for sid in base.sections}

    # --- collect -----------------------------------------------------------
    ordered = sorted(overlays, key=lambda f: f.fragment_id)   # one order, used everywhere
    replaced_by, appended_by = {}, {}
    for frag in ordered:
        for sid in frag.replaces:
            replaced_by.setdefault(sid, []).append(frag.fragment_id)
        for sid in frag.appends:
            appended_by.setdefault(sid, []).append(frag.fragment_id)

    # --- validate ----------------------------------------------------------
    for sid, claimants in sorted(replaced_by.items()):
        if sid not in content:
            raise CompileError(f"assembly.replace-unknown-section: '{sid}' ({claimants})")
        if len(claimants) > 1:
            raise CompileError(f"assembly.two-owners: '{sid}' claimed by {claimants}")
    for sid, claimants in sorted(appended_by.items()):
        if sid not in content:
            raise CompileError(f"assembly.append-unknown-section: '{sid}' ({claimants})")
        if sid in replaced_by:
            raise CompileError(f"assembly.append-and-replace: '{sid}' claimed by {replaced_by[sid] + claimants}")

    axis_values = {k: str(v) for k, v in axes.items()}
    values: Dict[str, str] = dict(axis_values)
    supplier: Dict[str, str] = {}
    for frag in ordered + [base]:
        for key, val in frag.param_values.items():
            if key in axis_values:
                raise CompileError(f"assembly.param-shadows-axis: '{key}' is an axis, redefined by {frag.fragment_id}")
            if key in supplier and values[key] != str(val):
                raise CompileError(
                    f"assembly.param-collision: '{key}' = {values[key]!r} ({supplier[key]}) "
                    f"vs {str(val)!r} ({frag.fragment_id})")
            values[key], supplier[key] = str(val), frag.fragment_id

    # --- apply -------------------------------------------------------------
    for frag in ordered:
        for sid in frag.replaces:
            content[sid], owner[sid] = frag.sections[sid], frag.fragment_id
            contributors[sid].append(frag.fragment_id)
    for frag in ordered:
        for sid in frag.appends:
            content[sid] = content[sid] + "\n\n" + frag.sections[sid]
            contributors[sid].append(frag.fragment_id)

    missing = [sid for sid in required if sid not in content]
    if missing:
        raise CompileError(f"assembly.required-section-unowned: {missing}")
    unclaimed = [sid for sid in base.must_replace if sid not in replaced_by]
    if unclaimed:
        raise CompileError(f"assembly.placeholder-not-replaced: {unclaimed}")

    unresolved: Set[str] = set()

    def _sub(m):
        key = m.group(1)
        if key not in values:
            unresolved.add(key)
            return m.group(0)
        return values[key]

    content = {sid: _PARAM.sub(_sub, text) for sid, text in content.items()}
    if unresolved:
        raise CompileError(f"assembly.unresolved-param: {sorted(unresolved)}")

    order = list(base.sections) + [s for s in content if s not in base.sections]
    rendered = "\n\n".join(content[sid] for sid in order).strip() + "\n"
    return Leaf(skill_id, dict(axes), order, content, owner, contributors,
                [{"fragment_id": f.fragment_id, "version": f.version,
                  "path": f.path, "source_sha256": f.source_sha256}
                 for f in [base, *ordered]],
                rendered, hashlib.sha256(rendered.encode()).hexdigest())


demo = compile_leaf(frags, {"tone": "formal", "audience": "developer"}, skill_id="demo")
print(f"{'section':24}{'final owner':22}contributors")
for sid in demo.order:
    print(f"{sid:24}{demo.owners[sid]:22}{demo.contributors[sid]}")

section                 final owner           contributors
output.contract         base-release-notes    ['base-release-notes', 'audience-developer']
summary.style           audience-developer    ['base-release-notes', 'audience-developer']
highlights.structure    tone-formal           ['base-release-notes', 'tone-formal']
address.contract        tone-formal           ['base-release-notes', 'tone-formal']
prohibitions.hard       tone-formal           ['base-release-notes', 'tone-formal']
typography              base-release-notes    ['base-release-notes']


In the example above, the base owns `output.contract`, while `audience-developer` appends instructions to it. The manifest needs to record both ownership and contributions so that we can trace a problematic rule back to the fragment that supplied it. Recording only the owner would incorrectly attribute all the section’s content to the base.

## 4. Emitting the compiled artifacts

Each compilation produces three artifacts:

- `<id>.md` contains the exact instruction text sent to the generator. Versions, hashes, and other metadata are stored separately to avoid adding unnecessary content to the model’s context.
- `<id>.sections.json` contains the same instructions organised by section ID, allowing the harness to select the relevant rubric sections at runtime.
- `manifest.json` records which fragments produced each compiled variant, including their versions and source hashes, each section’s owner and contributors, and the compiled variant’s content hash.

The content hash lets us check whether the instructions used in a run match those we reviewed. The source hashes identify the fragment contents used to compile them. Both are needed because a compiled file can remain unchanged after its source fragments have been edited. Comparing the recorded source hashes with the current fragments lets us detect when a rebuild is needed.

**Checking the intended variants.** `discover_axes` generates the Cartesian product of the axis values it finds. Successfully compiling every combination establishes that those combinations can be assembled, but doesn’t establish that all are intended for use. If only three of four combinations are supported, declare that list explicitly and compare it with the emitted variants.

In [ ]:
def discover_axes(fragments) -> List[Dict[str, str]]:
    values: Dict[str, Set[str]] = {}
    for frag in fragments:
        for key, want in frag.applies_when.items():
            got = want if isinstance(want, (list, tuple, set)) else [want]
            values.setdefault(key, set()).update(str(v) for v in got)
    combos = [{}]
    for key in [k for k in AXIS_ORDER if k in values]:
        combos = [{**c, key: v} for c in combos for v in sorted(values[key])]
    return combos


def leaf_id(axes) -> str:
    return "-".join(axes[k] for k in AXIS_ORDER if k in axes)


def build(src_root: Path, out_root: Path, required=()) -> List[Leaf]:
    fragments = load_fragments(src_root)
    leaves = [compile_leaf(fragments, axes, skill_id=leaf_id(axes), required=required)
              for axes in discover_axes(fragments)]
    out_root.mkdir(parents=True, exist_ok=True)
    for lf in leaves:
        (out_root / f"{lf.skill_id}.md").write_text(lf.content, encoding="utf-8")
        (out_root / f"{lf.skill_id}.sections.json").write_text(
            json.dumps({sid: lf.sections[sid] for sid in lf.order}, indent=2,
                       ensure_ascii=False) + "\n", encoding="utf-8")
    (out_root / "manifest.json").write_text(json.dumps({
        "leaves": [{"skill_id": lf.skill_id, "axes": lf.axes,
                    "content_sha256": lf.content_sha256,
                    "owners": lf.owners, "contributors": lf.contributors,
                    "fragments": lf.provenance} for lf in leaves]},
        indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return leaves


leaves = build(SRC, GENERATED, required=("output.contract",))
print(f"{'leaf':22}{'chars':>7}{'sections':>10}  sha256")
for lf in sorted(leaves, key=lambda x: x.skill_id):
    print(f"{lf.skill_id:22}{len(lf.content):>7}{len(lf.sections):>10}  {lf.content_sha256[:12]}")
print(f"\n{len(frags)} maintained fragments -> {len(leaves)} delivered leaves\n")

manifest = json.loads((GENERATED / "manifest.json").read_text())
entry = next(l for l in manifest["leaves"] if l["skill_id"] == "formal-developer")
print("manifest excerpt (formal-developer):")
print(json.dumps({"content_sha256": entry["content_sha256"][:16] + "...",
                  "contributors": {"output.contract": entry["contributors"]["output.contract"]},
                  "fragments": [{k: (v[:12] + "..." if k == "source_sha256" else v)
                                 for k, v in f.items() if k != "path"}
                                for f in entry["fragments"]]}, indent=2))

leaf                    chars  sections  sha256
casual-customer           923         6  691d37beb980
casual-developer         1000         6  67199f0a05a5
formal-customer           912         6  a0ed0c492cb4
formal-developer          989         6  b82ae305c858

5 maintained fragments -> 4 delivered leaves

manifest excerpt (formal-developer):
{
  "content_sha256": "b82ae305c858f43e...",
  "contributors": {
    "output.contract": [
      "base-release-notes",
      "audience-developer"
    ]
  },
  "fragments": [
    {
      "fragment_id": "base-release-notes",
      "source_sha256": "211dace84f4a...",
      "version": "1.0.0"
    },
    {
      "fragment_id": "audience-developer",
      "source_sha256": "49327f24ef1e...",
      "version": "1.0.0"
    },
    {
      "fragment_id": "tone-formal",
      "source_sha256": "f8c00f35d640...",
      "version": "1.0.0"
    }
  ]
}


In [ ]:
print((GENERATED / "formal-developer.md").read_text())

## 1. Output contract

Return one JSON object with exactly these keys:

- `summary`: one string.
- `highlights`: a list of strings.

No prose outside the JSON object. No key that is not listed here.

Add one further key:

- `breaking`: a boolean, true when a caller must change code.

## 2. Summary

One sentence, at most 25 words, written for a developer integrating
against the API. Name the affected endpoint or field when one changed.

## 3. Highlights

Each highlight is one bullet, starting with a noun phrase: `Improved export
throughput`. Between three and six bullets. One change per bullet.

## 4. Addressing the reader

Address the reader as `you`. Keep the verb form plain and unmarked:
`you can now export`. Never use a first-person plural subject.

## 5. Prohibitions

No exclamation marks. No superlatives (`best`, `fastest`, `most powerful`).
No promise about future releases.

## 6. Typography

Straight quotes. No trailing whitespace. Version numbers in digits: `2.4.0`.



## 5. Testing variant isolation before generation

When a single skill file contains every conditional branch, the model has to determine which instructions apply during generation. It may combine guidance from different branches, producing a document that mixes formal and casual styles. Even an instruction forbidding a word or phrase still places it in the model’s context.

Compilation lets us remove irrelevant branches before the model call. We can then test that a variant compiled for `casual` contains no formal-only instructions. This check runs directly on the compiled file without requiring a model call.

The build step makes part of variant isolation testable in code. We still need to evaluate the generated output, but we can detect conflicting instructions before they reach the model.

In [ ]:
formal = (GENERATED / "formal-customer.md").read_text()
casual = (GENERATED / "casual-customer.md").read_text()

assert "speak as `we`" not in formal
assert "Never use a first-person plural subject" in formal
assert "Never use a first-person plural subject" not in casual
for text in (formal, casual):
    assert "PLACEHOLDER" not in text and "{{" not in text

print("formal leaf, occurrences of the casual rule:", formal.count("speak as `we`"))
print("casual leaf, occurrences of the formal rule:",
      casual.count("Never use a first-person plural subject"))
print("\nboth leaves: no placeholder, no unresolved parameter, nothing to select at runtime")

formal leaf, occurrences of the casual rule: 0
casual leaf, occurrences of the formal rule: 0

both leaves: no placeholder, no unresolved parameter, nothing to select at runtime


## 6. What the compiler refuses to build

The following examples show defects the compiler catches before generation. Each causes the build to fail with an error identifying the relevant fragment and section, so we can correct the source without waiting for the problem to appear in generated output.

The error messages document the compiler’s validation rules for section ownership and parameter definitions.

In [ ]:
GOOD_BASE = '''---
fragment_id: base-x
kind: base
version: 1.0.0
provides: [output.contract, address.contract, address.footer]
must_replace: [address.contract]
---

<!-- section: output.contract -->
Return JSON.

<!-- section: address.contract -->
PLACEHOLDER.

<!-- section: address.footer -->
A footer.
'''

def tone(fid, section="address.contract", verb="replaces", params=None):
    p = f"param_values:\n  {params}\n" if params else ""
    return (f"---\nfragment_id: {fid}\nkind: tone\nversion: 1.0.0\n"
            f"applies_when:\n  tone: formal\n{verb}: [{section}]\n{p}---\n\n"
            f"<!-- section: {section} -->\nOwned by {fid}.\n")

def attempt(label, *texts, axes={"tone": "formal"}):
    try:
        compile_leaf([parse_fragment(t) for t in texts], axes, skill_id="t")
        print(f"{label:26} -> built (no error)")
    except CompileError as e:
        print(f"{label:26} -> {e}")

attempt("two owners",       GOOD_BASE, tone("tone-a"), tone("tone-b"))
attempt("append + replace", GOOD_BASE, tone("tone-a"), tone("tone-b", verb="appends"))
attempt("placeholder left", GOOD_BASE, axes={"tone": "casual"})
attempt("unknown section",  GOOD_BASE, tone("tone-a", section="address.nope"))
attempt("unresolved param", GOOD_BASE.replace("Return JSON.", "Return {{nobody}}."), tone("tone-a"))
attempt("param collision",  GOOD_BASE, tone("tone-a", params="noun: alpha"),
                            tone("tone-b", section="address.footer", params="noun: beta"))
attempt("param shadows axis", GOOD_BASE, tone("tone-a", params="tone: casual"))

for label, text in [
    ("namespace violation",   tone("tone-a", section="output.contract")),
    ("declared, not written", "---\nfragment_id: t\nkind: tone\nversion: 1.0.0\nreplaces: [address.contract]\n---\n\nno marker\n"),
    ("unknown axis",          "---\nfragment_id: t\nkind: tone\nversion: 1.0.0\napplies_when:\n  audiance: developer\nreplaces: [address.contract]\n---\n\n<!-- section: address.contract -->\nx\n"),
]:
    try:
        parse_fragment(text)
    except CompileError as e:
        print(f"{label:26} -> {e}")

two owners                 -> assembly.two-owners: 'address.contract' claimed by ['tone-a', 'tone-b']
append + replace           -> assembly.append-and-replace: 'address.contract' claimed by ['tone-a', 'tone-b']
placeholder left           -> assembly.placeholder-not-replaced: ['address.contract']
unknown section            -> assembly.replace-unknown-section: 'address.nope' (['tone-a'])
unresolved param           -> assembly.unresolved-param: ['nobody']
param collision            -> assembly.param-collision: 'noun' = 'alpha' (tone-a) vs 'beta' (tone-b)
param shadows axis         -> assembly.param-shadows-axis: 'tone' is an axis, redefined by tone-a
namespace violation        -> fragment.namespace-violation: kind 'tone' may not own 'output.contract' (tone-a)
declared, not written      -> fragment.declared-not-written: ['address.contract'] in t
unknown axis               -> fragment.unknown-axis: ['audiance'] in t (declared axes: ['tone', 'audience'])


## 7. Runtime: one artifact, several consumers

The section index lets each consumer access the instructions it needs from the same compiled artifact.

| Consumer | Reads | Purpose |
|---|---|---|
| Generator | The complete compiled variant | Apply all instructions during generation |
| Deterministic harness | Individual sections | Read vocabulary and other rule data directly from the skill |
| Validator re-ask | The violated section, verbatim | Return the original instruction to the model for correction |
| Judges | The sections relevant to the part being evaluated | Evaluate that part against its applicable rules |

Reading rules from the compiled artifact keeps generation, validation, and correction aligned. If a checker separately defines the same vocabulary in Python, both copies need to be maintained. When they diverge, the checker may reject output that follows the instructions the model received. This can trigger unnecessary retries and send developers looking for a generation error when the problem is in the checker.

In [ ]:
@lru_cache(maxsize=32)
def sections(tone_: str, audience: str) -> Dict[str, str]:
    '''Read the emitted artifact - never re-compile the sources at runtime.
    Fragments are a build-time input; a deployed package need not ship them.'''
    path = GENERATED / f"{tone_}-{audience}.sections.json"
    if not path.exists():
        raise FileNotFoundError(f"no compiled leaf for {tone_}-{audience}")
    return json.loads(path.read_text(encoding="utf-8"))


@lru_cache(maxsize=32)
def whole_leaf(tone_: str, audience: str) -> str:
    return (GENERATED / f"{tone_}-{audience}.md").read_text(encoding="utf-8")


def rule_quote(tone_: str, audience: str, section_id: str) -> str:
    '''One section, verbatim, for quoting back to the model in a re-ask.'''
    return sections(tone_, audience)[section_id]


TONE, AUD = "formal", "developer"      # the variant used for the rest of the notebook

print("addressable sections:", ", ".join(sorted(sections(TONE, AUD))), "\n")
print("--- rule_quote(TONE, AUD, 'prohibitions.hard') ---")
print(rule_quote(TONE, AUD, "prohibitions.hard"))

addressable sections: address.contract, highlights.structure, output.contract, prohibitions.hard, summary.style, typography 

--- rule_quote(TONE, AUD, 'prohibitions.hard') ---
## 5. Prohibitions

No exclamation marks. No superlatives (`best`, `fastest`, `most powerful`).
No promise about future releases.


## 8. Separating generated content, known facts, and contract checks

Judges evaluate the semantic rules for the parts assigned to them. Their approval doesn’t establish that the complete document satisfies the output contract. Each part could pass its evaluation while the assembled document still contains missing or unexpected fields.

The harness therefore handles three concerns separately:

| Kind | Example | Responsibility |
|---|---|---|
| Judged content | `summary`, `highlights` | The model generates it, and the panel evaluates it against the applicable skill rules |
| Harness-owned fact | `breaking` | The harness supplies the known value |
| Contract structure | The required set of keys | The executable schema defines it, and the harness checks agreement with the skill’s contract section |

**Known values are supplied by the harness.** Whether a release introduces breaking changes is provided as input in this example. The generation schema therefore omits `breaking`, and the harness adds it when assembling the delivery object. This avoids spending inference on a known value and removes the possibility of the model generating it incorrectly.

**The executable schema is checked against the skill.** The schemas below are written explicitly. The harness extracts the key names from the compiled `output.contract` section and checks that they match the delivery schema. This verifies agreement on field names; it doesn’t derive types or constraints from prose. Full schema derivation would require a machine-readable contract. The key check still catches changes made to one definition but omitted from the other.

In [ ]:
_CONTRACT_KEY = re.compile(r"^-\s+`([A-Za-z_][A-Za-z0-9_]*)`\s*:", re.M)

#: Facts the harness owns. Never generated, never judged, injected at assembly.
HARNESS_OWNED = ("breaking",)


def contract_keys(tone_: str, audience: str) -> Set[str]:
    '''The keys the compiled skill says the output must have.'''
    return set(_CONTRACT_KEY.findall(sections(tone_, audience)["output.contract"]))


class GeneratedNotes(BaseModel):
    '''What the MODEL is asked for: everything except what the harness knows.'''
    summary: str
    highlights: List[str] = Field(min_length=1)


class CustomerNotes(BaseModel):
    '''The delivery envelope for the customer variant.'''
    summary: str
    highlights: List[str] = Field(min_length=1)


class DeveloperNotes(BaseModel):
    '''The delivery envelope for the developer variant.'''
    summary: str
    highlights: List[str] = Field(min_length=1)
    breaking: bool


GENERATION_MODEL = {"customer": GeneratedNotes, "developer": GeneratedNotes}
ENVELOPE = {"customer": CustomerNotes, "developer": DeveloperNotes}


def check_schema_matches_contract(tone_: str, audience: str) -> None:
    '''Executable schema vs the skill's contract section. Not derivation - agreement.'''
    declared = contract_keys(tone_, audience)
    delivered = set(ENVELOPE[audience].model_fields)
    if declared != delivered:
        raise AssertionError(
            f"{tone_}-{audience}: skill declares {sorted(declared)}, envelope has "
            f"{sorted(delivered)} (missing={sorted(declared - delivered)}, "
            f"extra={sorted(delivered - declared)})")
    generated = set(GENERATION_MODEL[audience].model_fields)
    expected = declared - set(HARNESS_OWNED)
    if generated != expected:
        raise AssertionError(
            f"{tone_}-{audience}: generation schema {sorted(generated)} should be the "
            f"contract minus harness-owned fields {sorted(expected)}")


for t in ("formal", "casual"):
    for a in ("customer", "developer"):
        check_schema_matches_contract(t, a)
        owned = sorted(contract_keys(t, a) & set(HARNESS_OWNED))
        print(f"{t}-{a:10} contract {sorted(contract_keys(t, a))} "
              f"= generated {sorted(GENERATION_MODEL[a].model_fields)} + harness {owned}")

# What the check catches: a skill edit the schema did not follow.
_keep = ENVELOPE["customer"]
try:
    class StaleCustomerNotes(BaseModel):
        summary: str            # someone deleted `highlights` from the envelope
    ENVELOPE["customer"] = StaleCustomerNotes
    check_schema_matches_contract("formal", "customer")
except AssertionError as e:
    print("\ndrift caught ->", e)
finally:
    ENVELOPE["customer"] = _keep

formal-customer   contract ['highlights', 'summary'] = generated ['highlights', 'summary'] + harness []
formal-developer  contract ['breaking', 'highlights', 'summary'] = generated ['highlights', 'summary'] + harness ['breaking']
casual-customer   contract ['highlights', 'summary'] = generated ['highlights', 'summary'] + harness []
casual-developer  contract ['breaking', 'highlights', 'summary'] = generated ['highlights', 'summary'] + harness ['breaking']

drift caught -> formal-customer: skill declares ['highlights', 'summary'], envelope has ['summary'] (missing=['highlights'], extra=[])


## 9. Portioned judging: assigning rules to each output part

Each output part is mapped to the instruction sections that govern it. This mapping determines what the judges can evaluate and should be reviewed alongside the skill. Missing a relevant section leaves a requirement unchecked, while including unrelated sections adds unnecessary context and token cost.

The rubrics exclude `output.contract` because the schema validator handles the structural checks described above. Keeping those checks in code avoids an additional model call for requirements that can be evaluated deterministically.

In [ ]:
@dataclass(frozen=True)
class OutputSection:
    name: str
    fields: Tuple[str, ...]
    governed_by: Tuple[str, ...]


OUTPUT_SECTIONS: Tuple[OutputSection, ...] = (
    OutputSection("summary", ("summary",),
                  ("summary.style", "address.contract", "prohibitions.hard", "typography")),
    OutputSection("highlights", ("highlights",),
                  ("highlights.structure", "address.contract", "prohibitions.hard", "typography")),
)

AXES = ("rule_compliance", "specificity", "readability")
AXIS_WEIGHTS = {"rule_compliance": 0.5, "specificity": 0.3, "readability": 0.2}
AGREEMENT_THRESHOLD = 0.2      # judge std at which agreement collapses to 0
NEUTRAL = 0.5


def rubric(part: OutputSection, tone_: str, audience: str) -> str:
    '''The skill text governing one part - verbatim, in leaf order.'''
    index = sections(tone_, audience)
    missing = [sid for sid in part.governed_by if sid not in index]
    if missing:
        raise KeyError(f"rubric for '{part.name}' lost sections: {missing}")
    return "\n\n".join(index[sid] for sid in part.governed_by)


print(f"=== rubric for 'summary' ({TONE}-{AUD}) ===")
print(rubric(OUTPUT_SECTIONS[0], TONE, AUD))
print("\n>>> ABSENT on purpose: the output schema (validated, not judged),")
print(">>> and the highlights bullet rules (a different part's business).")

=== rubric for 'summary' (formal-developer) ===
## 2. Summary

One sentence, at most 25 words, written for a developer integrating
against the API. Name the affected endpoint or field when one changed.

## 4. Addressing the reader

Address the reader as `you`. Keep the verb form plain and unmarked:
`you can now export`. Never use a first-person plural subject.

## 5. Prohibitions

No exclamation marks. No superlatives (`best`, `fastest`, `most powerful`).
No promise about future releases.

## 6. Typography

Straight quotes. No trailing whitespace. Version numbers in digits: `2.4.0`.

>>> ABSENT on purpose: the output schema (validated, not judged),
>>> and the highlights bullet rules (a different part's business).


## 10. Deterministic gates: checking measurable rules

Code can check word counts and bullet limits directly. Judges are needed for questions such as whether the text is appropriate for a developer audience, where the answer requires interpretation.

The harness checks measurable requirements before sending candidates to the panel. These checks are authoritative for the rules they cover: a candidate that exceeds a word limit fails regardless of the panel’s assessment. Likewise, a judge’s estimate of the word count shouldn’t override a passing deterministic check. The panel then evaluates the remaining semantic requirements.

The harness reads each threshold and list from the skill, including word limits, bullet ranges, permitted exclamation marks, and prohibited superlatives. Keeping these values in the skill avoids maintaining a second definition in code. The harness implements the checking mechanisms, such as splitting text on whitespace, counting list items, and matching literal terms. Its number-word map supports parsing those values without defining the policy itself.

The gates only check what they explicitly cover. For example, `faster` isn’t included in the skill’s prohibited terms, so it won’t trigger a literal match. Whether its use amounts to self-praise remains a question for the judges.

In [ ]:
_WORD_LIMIT = re.compile(r"at most (\d+) words")
_BULLET_RANGE = re.compile(r"Between (\w+) and (\w+) bullets")
_NO_EXCLAMATION = re.compile(r"No exclamation marks")
_MAX_EXCLAMATION = re.compile(r"At most (\w+) exclamation mark")
_SUPERLATIVES = re.compile(r"No superlatives \(([^)]*)\)")
_BACKTICKED = re.compile(r"`([^`]+)`")
#: Mechanism, not policy: the numbers come from the skill, this only reads them.
_NUMBER_WORDS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
                 "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10}


class SkillRuleError(RuntimeError):
    '''The skill no longer states a threshold the gate enforces. Loud on purpose:
    a checker that silently stops checking looks exactly like clean output.'''


def _as_int(word: str) -> int:
    return int(word) if word.isdigit() else _NUMBER_WORDS[word.lower()]


def word_limit(tone_: str, audience: str) -> int:
    m = _WORD_LIMIT.search(sections(tone_, audience)["summary.style"])
    if not m:
        raise SkillRuleError("summary.style no longer states a word limit")
    return int(m.group(1))


def bullet_range(tone_: str, audience: str) -> Tuple[int, int]:
    m = _BULLET_RANGE.search(sections(tone_, audience)["highlights.structure"])
    if not m:
        raise SkillRuleError("highlights.structure no longer states a bullet range")
    return _as_int(m.group(1)), _as_int(m.group(2))


def exclamation_limit(tone_: str, audience: str) -> int:
    body = sections(tone_, audience)["prohibitions.hard"]
    if _NO_EXCLAMATION.search(body):
        return 0
    m = _MAX_EXCLAMATION.search(body)
    if not m:
        raise SkillRuleError("prohibitions.hard no longer states an exclamation rule")
    return _as_int(m.group(1))


def banned_superlatives(tone_: str, audience: str) -> Tuple[str, ...]:
    m = _SUPERLATIVES.search(sections(tone_, audience)["prohibitions.hard"])
    if not m:
        raise SkillRuleError("prohibitions.hard no longer lists superlatives")
    return tuple(_BACKTICKED.findall(m.group(1)))


def deterministic_problems(part_name: str, value, tone_: str, audience: str) -> List[str]:
    '''Measurable breaches, with the skill's own sentence attached to each.'''
    problems: List[str] = []
    text = value if isinstance(value, str) else "\n".join(value)

    if part_name == "summary":
        n, limit = len(str(value).split()), word_limit(tone_, audience)
        if n > limit:
            problems.append(f"{n} words, the skill allows at most {limit} -- "
                            f"{rule_quote(tone_, audience, 'summary.style').splitlines()[2]}")
    if part_name == "highlights":
        lo, hi = bullet_range(tone_, audience)
        if not lo <= len(value) <= hi:
            problems.append(f"{len(value)} bullets, the skill allows {lo} to {hi}")

    limit = exclamation_limit(tone_, audience)
    if text.count("!") > limit:
        problems.append(f"{text.count('!')} exclamation marks, the skill allows {limit}")
    for word in banned_superlatives(tone_, audience):
        if re.search(rf"\b{re.escape(word)}\b", text, re.I):
            problems.append(f"uses the superlative `{word}`, which the skill prohibits")
    return problems


print(f"thresholds read from the {TONE}-{AUD} leaf:")
print(f"   summary word limit : {word_limit(TONE, AUD)}")
print(f"   bullet range       : {bullet_range(TONE, AUD)}")
print(f"   exclamation limit  : {exclamation_limit(TONE, AUD)}")
print(f"   banned superlatives: {banned_superlatives(TONE, AUD)}")
print("\nand for the casual leaf, the same code reads different numbers:")
print(f"   exclamation limit  : {exclamation_limit('casual', AUD)}")

thresholds read from the formal-developer leaf:
   summary word limit : 25
   bullet range       : (3, 6)
   exclamation limit  : 0
   banned superlatives: ('best', 'fastest', 'most powerful')

and for the casual leaf, the same code reads different numbers:
   exclamation limit  : 1


## 11. Connecting real models

The following cells use live models through OpenRouter to generate candidates, evaluate them with a panel, and retry generation when corrections are needed.

Set `OPENROUTER_API_KEY` in your environment before running. If it’s missing, the cell prompts for it without displaying the value. A complete run costs a few cents. Section 16 calculates the cost from measured token usage and OpenRouter’s reported pricing.

**The panel uses different model families.** It includes one model each from Anthropic, OpenAI, and Google. Repeated samples from a single model may share the same blind spots, so their agreement can overstate the strength of the evaluation. Using different families adds diversity, although it doesn’t guarantee independent judgments.

**Model-specific request settings are kept in configuration.** For the routes used here, `google/gemini-3.1-flash-lite` uses Instructor’s JSON mode because it rejects TOOLS mode. The other two use TOOLS mode, with provider routing hints configured where needed. Adding another judge should require a configuration entry without changing the evaluation logic.

In [ ]:
import getpass, urllib.request
from typing import Annotated, Literal

import instructor
import openai
from pydantic import model_validator

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=False)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

BASE_URL = "https://openrouter.ai/api/v1"
GENERATOR = "openai/gpt-5.6-luna"

PANEL_MODELS = {
    "anthropic": dict(model="anthropic/claude-haiku-4.5",
                      mode=instructor.Mode.TOOLS,
                      extra_body={"provider": {"require_parameters": True}}),
    "openai":    dict(model="openai/gpt-5.4-mini",
                      mode=instructor.Mode.TOOLS,
                      extra_body={}),
    # TOOLS fails on this route ("no endpoints support tool_choice"); JSON works.
    "google":    dict(model="google/gemini-3.1-flash-lite",
                      mode=instructor.Mode.JSON,
                      extra_body={}),
}

USAGE: List[Dict[str, object]] = []          # every call, for the cost section


@lru_cache(maxsize=8)
def client_for(mode):
    return instructor.from_openai(
        openai.AsyncOpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url=BASE_URL),
        mode=mode)


async def typed_call(*, model, mode, extra_body, response_model, messages,
                     temperature=0.0, label="", max_retries=2):
    '''One structured call, with its usage recorded.'''
    t0 = time.perf_counter()
    out, completion = await client_for(mode).chat.completions.create_with_completion(
        model=model, response_model=response_model, max_retries=max_retries,
        temperature=temperature, extra_body=extra_body or None, messages=messages)
    u = getattr(completion, "usage", None)
    USAGE.append({"label": label or model, "model": model,
                  "prompt_tokens": getattr(u, "prompt_tokens", 0),
                  "completion_tokens": getattr(u, "completion_tokens", 0),
                  "seconds": time.perf_counter() - t0})
    return out


print(f"generator: {GENERATOR}")
for name, cfg in PANEL_MODELS.items():
    print(f"   {name:10} {cfg['model']:32} mode={cfg['mode'].value:12} extra_body={cfg['extra_body'] or '-'}")


generator: openai/gpt-5.6-luna
   anthropic  anthropic/claude-haiku-4.5       mode=tool_call    extra_body={'provider': {'require_parameters': True}}
   openai     openai/gpt-5.4-mini              mode=tool_call    extra_body=-
   google     google/gemini-3.1-flash-lite     mode=json_mode    extra_body=-

spend on this key before we start: $250.7358


### Generating candidates

The generator is the one consumer that gets the **whole leaf** — it needs every
rule. Its response model is `GeneratedNotes`: the contract minus the fields the
harness owns, so no inference is spent on `breaking`.

Three candidates at temperature 1.0. Group-relative judging needs a group, and a
group of near-identical candidates tells the panel nothing.

In [ ]:
RELEASE_INPUT = (
    "Release 2.4.0 of the Acme data platform.\n"
    "- the CSV exporter was rewritten; exports run about twice as fast\n"
    "- fixed escaping of quotes inside CSV fields\n"
    "- peak memory during export reduced by roughly 40 percent\n"
    "- the /v1/export endpoint now returns 202 with a job id instead of the file; "
    "callers must poll /v1/export/{id}\n"
)
RELEASE_FACTS = {"breaking": True}      # the harness knows; the model is never asked
N_CANDIDATES = 3


async def generate(n: int) -> List[dict]:
    async def one(i: int):
        out = await typed_call(
            model=GENERATOR, mode=instructor.Mode.TOOLS, extra_body={},
            response_model=GENERATION_MODEL[AUD], temperature=1.0, label=f"generate[{i}]",
            messages=[{"role": "system", "content": whole_leaf(TONE, AUD)},
                      {"role": "user", "content": RELEASE_INPUT}])
        return out.model_dump()
    return list(await asyncio.gather(*[one(i) for i in range(n)]))


CANDIDATES = await generate(N_CANDIDATES)
for i, c in enumerate(CANDIDATES):
    print(f"--- candidate {i}")
    print(f"  summary   : {c['summary']}")
    for h in c["highlights"]:
        print(f"  highlight : {h}")
    gates = {p.name: deterministic_problems(p.name, c[p.name], TONE, AUD) for p in OUTPUT_SECTIONS}
    for part_name, problems in gates.items():
        print(f"  gate/{part_name:11}: {'PASS' if not problems else 'FAIL — ' + '; '.join(problems)}")
    print()
print("fields the harness owns and never generated:", dict(RELEASE_FACTS))

--- candidate 0
  summary   : Release 2.4.0 improves CSV export performance and changes /v1/export to asynchronous job polling.
  highlight : CSV export throughput is about twice as fast.
  highlight : CSV quote escaping is corrected.
  highlight : Export peak memory usage is reduced by roughly 40 percent.
  highlight : The /v1/export endpoint now returns 202 with a job id, and you must poll /v1/export/{id} for the file.
  gate/summary    : PASS
  gate/highlights : PASS

--- candidate 1
  summary   : Release 2.4.0 changes CSV export performance and makes /v1/export asynchronous.
  highlight : CSV export runs about twice as fast
  highlight : CSV quote escaping is fixed
  highlight : Peak export memory is reduced by roughly 40 percent
  highlight : The /v1/export response now returns 202 with a job ID
  highlight : Callers poll /v1/export/{id} for the completed file
  gate/summary    : PASS
  gate/highlights : PASS

--- candidate 2
  summary   : Release 2.4.0 rewrites CSV export and cha

## 12. The panel: relative scoring and agreement

The panel follows the RULER-style relative scoring approach used by `agensflow`’s `relative_judge`:

- Each judge evaluates the candidates together, allowing direct comparison within the group. This reduces reliance on absolute scoring scales, although rankings can still differ between judges.
- Each judge returns scores for the configured evaluation axes. The harness combines them into a single score using the configured axis weights.
- The panel score is the mean of the individual judge scores. Their population standard deviation measures disagreement, with `agreement = 1 - clamp(std / threshold, 0, 1)`.
- Standard deviation is also calculated for each axis, helping us identify which requirements produced different assessments. Disagreement may indicate an ambiguous rubric or skill rule and gives us a specific place to investigate.
- If a judge fails or omits a candidate, the harness assigns a configured neutral score rather than calculating the result from a smaller panel.

**Agreement measures consistency, not correctness.** Judges can agree and still make the same mistake. Using different model families may reduce shared blind spots, but it doesn’t eliminate them. High agreement means the panel produced similar scores; it doesn’t establish that the output is correct.

The response schema also constrains what counts as a valid evaluation. `verdict` accepts only `"OK"` or `"VIOLATION"`, so an answer such as `"UNCERTAIN"` triggers a validation error and an Instructor re-ask. Axis names are restricted to the configured axes, scores must fall within `[0,1]`, and all three axes are required. Every candidate index must appear exactly once. Missing or duplicate candidates make the response invalid and trigger the same validation process.

In [ ]:
VERDICT_OK, VERDICT_VIOLATION = "OK", "VIOLATION"
Axis = Literal["rule_compliance", "specificity", "readability"]
Unit = Annotated[float, Field(ge=0.0, le=1.0)]


class _Verdict(BaseModel):
    candidate: int = Field(ge=0)
    score: Unit
    verdict: Literal["OK", "VIOLATION"]
    explanation: str = Field(min_length=1)
    axis_scores: Dict[Axis, Unit]

    @field_validator("axis_scores")
    @classmethod
    def all_axes_present(cls, v):
        missing = set(AXES) - set(v)
        if missing:
            raise ValueError(f"axis_scores is missing {sorted(missing)}")
        return v


@lru_cache(maxsize=8)
def judgement_model(n: int):
    '''Response model for a group of n candidates: exactly one verdict each.
    Baking n into the schema turns "the judge skipped one" into a validation
    error the library re-asks with, instead of a hole we paper over later.'''
    class _Judgement(BaseModel):
        scores: List[_Verdict]

        @model_validator(mode="after")
        def one_verdict_per_candidate(self):
            got = sorted(v.candidate for v in self.scores)
            if got != list(range(n)):
                raise ValueError(f"expected one verdict per candidate 0..{n - 1}, got {got}")
            return self
    return _Judgement


@dataclass(frozen=True)
class JudgeVerdict:
    candidate: int
    score: float
    verdict: str
    explanation: str
    axis_scores: Dict[str, float] = field(default_factory=dict)


@dataclass(frozen=True)
class GroupScore:
    candidate: int
    score: float                                  # mean of per-judge composed scalars
    verdict: str                                  # majority, ties -> VIOLATION
    agreement: float                              # 1 - clamp(std / threshold, 0, 1)
    disagreement_std: float
    disagreement_range: float
    per_judge_scores: Dict[str, float]
    per_judge_verdicts: Dict[str, str]
    axis_scores: Dict[str, float]
    per_axis_std: Dict[str, float]
    violations: List[str]


JudgeFn = Callable[[str, str, List[str]], Awaitable[List[JudgeVerdict]]]


def judge_system(part: OutputSection, n: int, rules: str) -> str:
    '''Mechanism only. Every RULE in this prompt comes from the compiled skill.'''
    return (f"You judge the `{part.name}` part of {n} candidate documents, relative to each "
            f"other. Score each on the axes {list(AXES)}, each in [0,1]. Set `verdict` to "
            f"OK if the version breaks no rule, or VIOLATION if it breaks at least one. In "
            f"the explanation, quote the rule broken - never a rule the version satisfies. "
            f"Word counts, bullet counts and prohibited literals are already verified "
            f"mechanically; judge meaning, not arithmetic.\n\n"
            f"RULES (verbatim from the skill):\n{rules}")


def make_judge(name: str, cfg: dict) -> JudgeFn:
    async def judge(system: str, user: str, candidates: List[str]) -> List[JudgeVerdict]:
        listed = "\n\n".join(f"### candidate {i}\n{c}" for i, c in enumerate(candidates))
        try:
            out = await typed_call(
                model=cfg["model"], mode=cfg["mode"], extra_body=cfg["extra_body"],
                response_model=judgement_model(len(candidates)), label=f"judge:{name}",
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": f"{user}\n\n{listed}".strip()}])
        except Exception as e:
            # Fail closed: a judge that cannot answer is a missing opinion, never
            # an implicit approval.
            print(f"   ! judge {name} failed: {type(e).__name__}: {str(e)[:90]}")
            return []
        return [JudgeVerdict(v.candidate, v.score, v.verdict, v.explanation, dict(v.axis_scores))
                for v in out.scores]
    return judge


PANEL: Dict[str, JudgeFn] = {n: make_judge(n, c) for n, c in PANEL_MODELS.items()}


def _compose(axis_scores: Dict[str, float], holistic: float) -> float:
    overlap = {k: AXIS_WEIGHTS[k] for k in axis_scores if k in AXIS_WEIGHTS}
    total = sum(overlap.values())
    if not overlap or total <= 0:
        return holistic
    return sum(axis_scores[k] * w for k, w in overlap.items()) / total


def aggregate(per_judge: Dict[str, List[JudgeVerdict]], n_candidates: int) -> List[GroupScore]:
    out: List[GroupScore] = []
    for i in range(n_candidates):
        effective, verdicts, explanations, axes_by_judge = {}, {}, [], {}
        for judge_name, verdict_list in per_judge.items():
            v = next((x for x in verdict_list if x.candidate == i), None)
            if v is None:
                effective[judge_name] = NEUTRAL
                verdicts[judge_name] = VERDICT_VIOLATION
                explanations.append(f"({judge_name} returned no verdict)")
                continue
            effective[judge_name] = _compose(v.axis_scores, v.score)
            verdicts[judge_name] = v.verdict
            axes_by_judge[judge_name] = v.axis_scores
            if v.verdict == VERDICT_VIOLATION:
                explanations.append(f"[{judge_name}] {v.explanation}")

        vals = list(effective.values())
        mean = sum(vals) / len(vals)
        std = statistics.pstdev(vals) if len(vals) > 1 else 0.0
        agreement = max(0.0, min(1.0, 1.0 - std / max(AGREEMENT_THRESHOLD, 1e-9)))

        axis_mean, axis_std = {}, {}
        for axis in sorted({a for d in axes_by_judge.values() for a in d}):
            got = [d[axis] for d in axes_by_judge.values() if axis in d]
            axis_mean[axis] = sum(got) / len(got)
            axis_std[axis] = statistics.pstdev(got) if len(got) > 1 else 0.0

        n_violation = sum(1 for v in verdicts.values() if v == VERDICT_VIOLATION)
        verdict = VERDICT_VIOLATION if n_violation * 2 >= len(verdicts) else VERDICT_OK

        out.append(GroupScore(i, mean, verdict, agreement, std, max(vals) - min(vals),
                              effective, verdicts, axis_mean, axis_std, explanations))
    return out


async def score_group(panel, system: str, user: str, candidates: List[str]) -> List[GroupScore]:
    names = list(panel)
    results = await asyncio.gather(*[panel[n](system, user, candidates) for n in names])
    return aggregate(dict(zip(names, results)), len(candidates))


def as_text(value) -> str:
    return value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, indent=2)


part = OUTPUT_SECTIONS[0]
eligible = [i for i, c in enumerate(CANDIDATES)
            if not deterministic_problems(part.name, c[part.name], TONE, AUD)]
print(f"gate: {len(eligible)}/{len(CANDIDATES)} candidates measurably clean -> judged\n")

t0 = time.perf_counter()
group = await score_group(PANEL, judge_system(part, len(eligible), rubric(part, TONE, AUD)), "",
                          [as_text(CANDIDATES[i][part.name]) for i in eligible])
print(f"panel returned in {time.perf_counter() - t0:.1f}s\n")

for g in group:
    print(f"candidate {eligible[g.candidate]}: score={g.score:.2f}  verdict={g.verdict:9} "
          f"agreement={g.agreement:.2f}  std={g.disagreement_std:.3f}")
    print(f"   per judge : { {k: round(v, 2) for k, v in g.per_judge_scores.items()} }")
    print(f"   verdicts  : {g.per_judge_verdicts}")
    print(f"   axis std  : { {k: round(v, 2) for k, v in g.per_axis_std.items()} }")
    for v in g.violations:
        print(f"   ! {v[:150]}")

gate: 3/3 candidates measurably clean -> judged

panel returned in 4.9s

candidate 0: score=0.85  verdict=OK        agreement=0.64  std=0.073
   per judge : {'anthropic': 0.86, 'openai': 0.93, 'google': 0.75}
   verdicts  : {'anthropic': 'OK', 'openai': 'OK', 'google': 'VIOLATION'}
   axis std  : {'readability': 0.07, 'rule_compliance': 0.24, 'specificity': 0.14}
   ! [google] The summary fails to address the reader as 'you' as required by rule 4.
candidate 1: score=0.82  verdict=OK        agreement=0.52  std=0.097
   per judge : {'anthropic': 0.92, 'openai': 0.85, 'google': 0.69}
   verdicts  : {'anthropic': 'OK', 'openai': 'OK', 'google': 'VIOLATION'}
   axis std  : {'readability': 0.08, 'rule_compliance': 0.24, 'specificity': 0.07}
   ! [google] The summary fails to address the reader as 'you' as required by rule 4.
candidate 2: score=0.81  verdict=OK        agreement=0.77  std=0.046
   per judge : {'anthropic': 0.85, 'openai': 0.84, 'google': 0.75}
   verdicts  : {'anthropic': 'OK'

### Sidebar: what skill compilation makes deterministic

Compiling a skill makes instruction assembly deterministic. It doesn’t make generation or model-based evaluation deterministic. The harness needs to account for that distinction.

| Deterministic for fixed inputs and configuration | Subject to model or runtime variation |
|---|---|
| Compiled instruction text and its hash | The sentences the generator produces |
| Included and excluded instruction sections | How the model applies those instructions |
| Section index and assembled rubrics | A judge’s interpretation of a rule |
| Schema checks and agreement with contract keys | Scores, verdicts, and candidate rankings |
| Envelope validation, coverage checks, and fact comparisons | Whether a repair improves the output |
| Retry limits and permitted state transitions | Latency and cost of individual calls |

The panel results illustrate this distinction. The three judges received the same rubric from the same compiled instructions and evaluated the same candidate text. They still differed in their verdicts, axis scores, and explanations. Consistent instructions make those differences easier to investigate, but don’t prevent them.

The word-count requirement provides a concrete example. Before the deterministic gate in Section 10 was added, the panel evaluated whether summaries met the 25-word limit. Across runs, judges described an 18-word summary as containing 28 words. Other responses reported counts of 14 or 18 words while still claiming that the summary exceeded the limit. Some also identified a separate problem—the summary didn’t name an endpoint—but combined that observation with an incorrect word-count verdict.

These responses show why we need to separate measurable requirements from semantic assessment. A judge may identify a relevant issue while giving an explanation that doesn’t support its verdict.

Repeated evaluation also produced different outcomes. In the captured run, the panel in Section 12 returned a majority `VIOLATION` for all three summaries. When called again on the same texts in Section 13, it returned a majority `OK` for one of them. The rubric and candidates were unchanged, but the verdict changed. Majority voting and the reported agreement measure help us assess this variation; neither guarantees a correct result.

The deterministic gate therefore runs first and is authoritative for the requirements it checks. It reads thresholds from the skill and applies the corresponding checks in code. The panel evaluates requirements that need interpretation, such as whether the wording is appropriate for a developer audience.

> **Check measurable requirements in code, and use judges for requirements that need interpretation.** Read thresholds from the skill so that the checker and the instructions use the same definition.

For example, `word_limit()` parses “at most N words” from `summary.style`, keeping the limit in the maintained instructions rather than duplicating it in Python.

Compilation gives us a stable, inspectable instruction input. If two runs have the same compiled instruction hash but different outputs, we can rule out a change in that instruction text. Investigating the difference still requires the remaining request data, model configuration, and execution records.

This is why the harness combines panel verdicts and agreement measures with deterministic validation, bounded repair, and explicit terminal states. These mechanisms let us manage model variability while keeping the surrounding execution rules testable.

Look at the disagreement alongside the scores. `per_axis_std` identifies which evaluation axes produced different assessments. Since the deterministic gate already handles measurable requirements, these differences help us investigate how the judges interpret the remaining semantic rules.

The exact scores and verdicts may change when you rerun the notebook. A panel lets us observe variation across judges alongside the aggregate result, although agreement alone doesn’t establish correctness.

## 13. The repair loop and its validation rules

**0. Check measurable requirements before judging.** Candidates that fail a deterministic check are excluded from the panel evaluation. If all candidates fail, the harness skips judging and requests a repair using the detected violations as feedback.

**1. Use verdicts to determine eligibility and scores to rank candidates.** The harness first keeps candidates that pass the panel’s verdict, then selects the highest-scoring candidate from that set. This prevents a higher score from taking precedence over a failed validation.

**2. Repair only the failing part.** The repair request includes the dissenting judges’ feedback verbatim. Parts that already passed remain unchanged, avoiding unnecessary regeneration and the possibility of introducing new errors in them.

**3. Compare the repair with the original.** The repair must pass the deterministic gate before entering the panel evaluation. Eligible original and repaired candidates are then judged together using the same rubric.

**4. Stop when the repair limit is reached.** If the part still fails validation after the permitted attempts, the harness records `failed_validation` and marks the document as undeliverable. This makes the failure explicit rather than returning an artifact that failed the required checks as a successful result.

In [ ]:
STATUS_OK, STATUS_REPAIRED, STATUS_FAILED = "ok", "repaired", "failed_validation"
MIN_AGREEMENT = 0.5
MAX_REPAIR_ATTEMPTS = 1


class SummaryPart(BaseModel):
    summary: str

    @field_validator("summary")
    @classmethod
    def within_skill_word_limit(cls, v):
        if len(v.split()) > word_limit(TONE, AUD):
            raise ValueError(rule_quote(TONE, AUD, "summary.style"))
        return v


class HighlightsPart(BaseModel):
    highlights: List[str] = Field(min_length=1)


PART_MODELS = {"summary": SummaryPart, "highlights": HighlightsPart}


@dataclass
class PartOutcome:
    part: str
    chosen: object
    score: float
    agreement: float
    status: str
    detail: str
    attempts: int = 0
    review: bool = False        # passed, but the panel barely agreed
    gate_rejected: int = 0      # candidates the deterministic gate removed


def select(group: List[GroupScore]) -> Tuple[GroupScore, bool]:
    '''Lexicographic: compliant candidates first, score only ranks within them.'''
    passing = [g for g in group if g.verdict == VERDICT_OK]
    pool = passing or group
    return max(pool, key=lambda g: g.score), bool(passing)


def retry_system(part: OutputSection, rules: str, reasons: List[str]) -> str:
    return (f"Rewrite only the `{part.name}` part of the release notes. It was rejected "
            f"for these reasons:\n" + "\n".join(f"- {r}" for r in reasons) +
            f"\n\nRULES (verbatim from the skill):\n{rules}")


async def repair(part: OutputSection, reasons: List[str], rules: str, current):
    out = await typed_call(
        model=GENERATOR, mode=instructor.Mode.TOOLS, extra_body={},
        response_model=PART_MODELS[part.name], temperature=0.2, label=f"repair:{part.name}",
        messages=[{"role": "system", "content": retry_system(part, rules, reasons)},
                  {"role": "user", "content": f"{RELEASE_INPUT}\n\nCurrent version:\n{as_text(current)}"}])
    return getattr(out, part.name)


async def judge_part(part, candidates, tone_, audience, *, panel, rewrite=repair, user_input=""):
    rules = rubric(part, tone_, audience)
    versions = [c[part.name] for c in candidates]
    gates = [deterministic_problems(part.name, v, tone_, audience) for v in versions]
    eligible = [i for i, g in enumerate(gates) if not g]
    rejected = len(versions) - len(eligible)

    if eligible:
        group = await score_group(panel, judge_system(part, len(eligible), rules), user_input,
                                  [as_text(versions[i]) for i in eligible])
        best, had_pass = select(group)
        current = versions[eligible[best.candidate]]
        if had_pass:
            return PartOutcome(part.name, current, best.score, best.agreement, STATUS_OK,
                               "gate passed, panel majority OK", 0,
                               review=best.agreement < MIN_AGREEMENT, gate_rejected=rejected)
        reasons, score, agreement = best.violations, best.score, best.agreement
    else:
        # No judge call: nothing semantic to weigh while a measurable rule is broken.
        base = min(range(len(versions)), key=lambda i: len(gates[i]))
        current, reasons, score, agreement = versions[base], gates[base], 0.0, 1.0
        print(f"   [{part.name}] every candidate failed the gate; repairing without judging")

    if rewrite is None:
        return PartOutcome(part.name, current, score, agreement, STATUS_FAILED,
                           "no candidate passed, no rewriter configured", 0, gate_rejected=rejected)

    attempts, detail = 0, "; ".join(reasons)
    while attempts < MAX_REPAIR_ATTEMPTS:
        attempts += 1
        try:
            fixed = await rewrite(part, reasons, rules, current)
        except Exception as e:
            detail = f"repair call failed: {type(e).__name__}: {str(e)[:80]}"
            break

        still = deterministic_problems(part.name, fixed, tone_, audience)
        if still:                      # the gate applies to repairs too
            reasons, detail = still, "; ".join(still)
            continue

        pair = [current, fixed] if not deterministic_problems(part.name, current, tone_, audience) else [fixed]
        regroup = await score_group(panel, judge_system(part, len(pair), rules), user_input,
                                    [as_text(v) for v in pair])
        winner, had_pass = select(regroup)
        chosen = pair[winner.candidate]
        if had_pass:
            return PartOutcome(part.name, chosen, winner.score, winner.agreement,
                               STATUS_REPAIRED if chosen is fixed else STATUS_OK,
                               "gate passed, panel accepted after repair", attempts,
                               review=winner.agreement < MIN_AGREEMENT, gate_rejected=rejected)
        current, reasons, detail = chosen, winner.violations, "; ".join(winner.violations)

    return PartOutcome(part.name, current, score, agreement, STATUS_FAILED,
                       f"still rejected after {attempts} attempt(s): {detail}", attempts,
                       gate_rejected=rejected)


async def judge_and_assemble(candidates, tone_, audience, *, panel, rewrite=repair,
                             facts=None, user_input=""):
    outcomes = await asyncio.gather(*[
        judge_part(p, candidates, tone_, audience, panel=panel, rewrite=rewrite,
                   user_input=user_input) for p in OUTPUT_SECTIONS])
    document = {o.part: o.chosen for o in outcomes}
    # Harness-owned facts are injected, not generated and not judged.
    for key in HARNESS_OWNED:
        if facts and key in facts:
            document[key] = facts[key]
    return {"document": document, "outcomes": list(outcomes),
            "deliverable": all(o.status != STATUS_FAILED for o in outcomes)}


t0 = time.perf_counter()
result = await judge_and_assemble(CANDIDATES, TONE, AUD, panel=PANEL, facts=RELEASE_FACTS)
print(f"\nfull round in {time.perf_counter() - t0:.1f}s\n")

for o in result["outcomes"]:
    flag = "  [review: low agreement]" if o.review else ""
    print(f"{o.part:12} status={o.status:16} gate_rejected={o.gate_rejected} attempts={o.attempts} "
          f"score={o.score:.2f} agreement={o.agreement:.2f}{flag}")
    print(f"{'':12} {o.detail[:150]}")

print(f"\ndeliverable: {result['deliverable']}")
print(json.dumps(result["document"], indent=2, ensure_ascii=False))


full round in 4.9s

summary      status=ok               gate_rejected=0 attempts=0 score=0.85 agreement=0.64
             gate passed, panel majority OK
highlights   status=ok               gate_rejected=0 attempts=0 score=0.94 agreement=0.91
             gate passed, panel majority OK

deliverable: True
{
  "summary": "Release 2.4.0 improves CSV export performance and changes /v1/export to asynchronous job polling.",
  "highlights": [
    "CSV export throughput approximately doubled.",
    "CSV quote escaping is corrected.",
    "Peak export memory usage decreased by approximately 40 percent.",
    "The /v1/export endpoint now returns 202 with a job id, and you can poll /v1/export/{id} for the file."
  ],
  "breaking": true
}


## 14. After assembly: deterministic validation and integration checks

The assembled document has never existed before — parts from different candidates
plus harness facts. Every guarantee that held for a *candidate* is re-established
for the *assembly*:

1. **Envelope validation** — the delivery schema (proved in section 8 to agree
   with the skill's contract) validates the assembled object.
2. **Coverage** — every contract key present, nothing extra along for the ride.
3. **Harness-owned fields** — the injected value is what the harness knows.

With injection rather than echo, the third check cannot fail at assembly time,
and that is the point: the failure mode is *designed out* rather than detected.
It stays because assembly is not the last thing that will ever touch this
document, and a check that costs nothing is cheap insurance against the step
someone adds next quarter.

In [ ]:
def validate_assembly(document, tone_, audience, facts):
    problems = []
    try:
        ENVELOPE[audience].model_validate(document)
    except ValidationError as e:
        problems += [f"schema: {err['loc']}: {err['msg']}" for err in e.errors()]

    declared, present = contract_keys(tone_, audience), set(document)
    if declared - present:
        problems.append(f"coverage: missing {sorted(declared - present)}")
    if present - declared:
        problems.append(f"coverage: unexpected {sorted(present - declared)}")

    for key, value in (facts or {}).items():
        if document.get(key) != value:
            problems.append(f"fact: '{key}' is {document.get(key)!r}, harness knows {value!r}")
    return problems


print("assembled document ->", validate_assembly(result["document"], TONE, AUD, RELEASE_FACTS) or "valid")

broken = dict(result["document"], highlights="- one string, not a list")
print("\na later step flattened highlights to a string ->")
for p in validate_assembly(broken, TONE, AUD, RELEASE_FACTS):
    print("   ", p)

lying = dict(result["document"], breaking=False)
print("\na later step overwrote a harness-owned fact ->")
for p in validate_assembly(lying, TONE, AUD, RELEASE_FACTS):
    print("   ", p)

assembled document -> valid

a later step flattened highlights to a string ->
    schema: ('highlights',): Input should be a valid list

a later step overwrote a harness-owned fact ->
    fact: 'breaking' is False, harness knows True


## 15. Generation per part

The pipeline already evaluates and repairs each output part separately. Judges receive the relevant rubric, deterministic gates apply the corresponding thresholds, and repair requests include only the failing part and its rules. Generation still receives the complete compiled instructions and produces every field in one response.

Splitting generation by part extends the same approach to the initial model calls. It lets us:

- **Limit each call to relevant instructions.** A call generating the summary receives its style and content requirements without the bullet-count rules and examples for the highlights.
- **Generate and retry parts independently.** The harness can request another candidate for one part while retaining accepted content elsewhere, as the repair loop already does.
- **Choose models by part.** Writing a short summary and covering several changes accurately impose different requirements. Separate calls let us evaluate which model is appropriate for each.

The existing `rubric(part)` provides the instruction sections for each call. We also need a contract slice specifying the single output key that call must return.

The resulting parts are assembled into the same candidate dictionary used throughout the pipeline. `judge_part`, the deterministic gates, the repair loop, and final assembly continue to work with that structure.

In [ ]:
_CONTRACT_BULLET = re.compile(r"^-\s+`([A-Za-z_][A-Za-z0-9_]*)`\s*:", re.M)


def contract_slice(tone_: str, audience: str, fields: Sequence[str]) -> str:
    """The output contract, keeping only the keys this part returns.

    The framing lines ("Return one JSON object", "no prose outside it") stay:
    they are rules about the response, not about a field. Only the key bullets
    are filtered — so a part is told exactly what it owns and nothing else.
    """
    kept = []
    for line in sections(tone_, audience)["output.contract"].splitlines():
        m = _CONTRACT_BULLET.match(line)
        if m and m.group(1) not in fields:
            continue
        kept.append(line)
    return "\n".join(kept).strip()


def generation_surface(part: OutputSection, tone_: str, audience: str) -> str:
    """Everything the model needs to write ONE part, and nothing else."""
    return (f"You write only the `{part.name}` part of a release note.\n\n"
            f"{contract_slice(tone_, audience, part.fields)}\n\n"
            f"RULES (verbatim from the skill):\n{rubric(part, tone_, audience)}")


def surface_digest(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# The cost section (21) counts tokens for its own table; this is the same
# counter, defined early because the comparison starts here.
try:
    import tiktoken
    _e = tiktoken.get_encoding("cl100k_base")
    ntok_ = lambda t: len(_e.encode(t))
except Exception:
    ntok_ = lambda t: max(1, round(len(t) / 4))

whole = ntok_(whole_leaf(TONE, AUD))
print(f"{'generation call':<16}{'tokens':>8}{'share of whole leaf':>22}  surface digest")
for p in OUTPUT_SECTIONS:
    s = generation_surface(p, TONE, AUD)
    print(f"{p.name:<16}{ntok_(s):>8}{ntok_(s) / whole:>21.0%}  {surface_digest(s)}")
print(f"{'whole document':<16}{whole:>8}{1.0:>21.0%}  {surface_digest(whole_leaf(TONE, AUD))}")

print("\n--- generation surface for 'summary' ---")
print(generation_surface(OUTPUT_SECTIONS[0], TONE, AUD))


In [ ]:
WALL: Dict[str, float] = {}          # wall clock per phase, for section 18


async def generate_part(part: OutputSection, i: int, *, route=None, temperature=1.0):
    """One candidate for one part. The route decides which model writes it."""
    route = route or {"model": GENERATOR, "mode": instructor.Mode.TOOLS, "extra_body": {}}
    out = await typed_call(
        model=route["model"], mode=route["mode"], extra_body=route["extra_body"],
        response_model=PART_MODELS[part.name], temperature=temperature,
        label=f"split:{part.name}[{i}]",
        messages=[{"role": "system", "content": generation_surface(part, TONE, AUD)},
                  {"role": "user", "content": RELEASE_INPUT}])
    return getattr(out, part.name)


async def generate_split(n: int, routes=None) -> List[dict]:
    """parts x candidates, all in flight at once, zipped back into candidates.

    A candidate is still a dict keyed by part, which is what every downstream
    consumer already expects — so nothing after this line needed changing.
    """
    routes = routes or {}
    jobs = [generate_part(p, i, route=routes.get(p.name))
            for p in OUTPUT_SECTIONS for i in range(n)]
    flat = await asyncio.gather(*jobs)
    per_part = {p.name: flat[k * n:(k + 1) * n] for k, p in enumerate(OUTPUT_SECTIONS)}
    return [{p.name: per_part[p.name][i] for p in OUTPUT_SECTIONS} for i in range(n)]


t0 = time.perf_counter()
SPLIT_CANDIDATES = await generate_split(N_CANDIDATES)
WALL["split generation"] = time.perf_counter() - t0

print(f"{len(OUTPUT_SECTIONS)} parts x {N_CANDIDATES} candidates = "
      f"{len(OUTPUT_SECTIONS) * N_CANDIDATES} calls in "
      f"{WALL['split generation']:.1f}s wall clock\n")
for i, c in enumerate(SPLIT_CANDIDATES):
    print(f"--- candidate {i}")
    print(f"  summary   : {c['summary']}")
    for h in c["highlights"]:
        print(f"  highlight : {h}")
    for p in OUTPUT_SECTIONS:
        problems = deterministic_problems(p.name, c[p.name], TONE, AUD)
        print(f"  gate/{p.name:11}: {'PASS' if not problems else 'FAIL — ' + '; '.join(problems)}")
    print()


## 16. Choosing a model per part

Generating each part in a separate call also lets us choose a model for that part. The requirements differ, so we can evaluate model suitability separately.

| Part | Main requirements | Failures to evaluate |
|---|---|---|
| `summary` | Produce one sentence within the word limit and avoid prohibited language | Exceeding the limit or using terms such as “significantly” |
| `highlights` | Cover every supplied change, add no unsupported claims, and follow the bullet format | Combining changes in a way that loses coverage or inventing an additional change |

`PART_ROUTES` maps each part name to a model. The selected route is recorded with the call, alongside the compiled instructions.

This example demonstrates the routing and evaluation method. Three candidates per part for a single input aren’t enough to establish which model performs better. That comparison requires the full case set, repeated runs, and evaluation against the applicable rules. The export notebook supports that broader analysis.

In [ ]:
#: Which model writes which part. A dict entry, never a code change.
PART_ROUTES = {
    "summary":    dict(model="openai/gpt-5.4-mini", mode=instructor.Mode.TOOLS, extra_body={}),
    "highlights": dict(model=GENERATOR,             mode=instructor.Mode.TOOLS, extra_body={}),
}

for name, route in PART_ROUTES.items():
    print(f"{name:12} -> {route['model']}")

t0 = time.perf_counter()
ROUTED_CANDIDATES = await generate_split(N_CANDIDATES, routes=PART_ROUTES)
WALL["routed generation"] = time.perf_counter() - t0
print(f"\nrouted generation: {WALL['routed generation']:.1f}s wall clock\n")


def gate_summary(candidates, label):
    """How often each part cleared the deterministic gate — the one comparison
    that needs no judge and no interpretation."""
    for p in OUTPUT_SECTIONS:
        bad = [deterministic_problems(p.name, c[p.name], TONE, AUD) for c in candidates]
        clean = sum(1 for b in bad if not b)
        first = next((b[0] for b in bad if b), "")
        print(f"   {label:>10} {p.name:<12} gate {clean}/{len(candidates)} clean"
              + (f"   e.g. {first[:70]}" if first else ""))


print("gate pass rate, same model everywhere vs routed:")
gate_summary(SPLIT_CANDIDATES, "one model")
gate_summary(ROUTED_CANDIDATES, "routed")

for i, c in enumerate(ROUTED_CANDIDATES[:1]):
    print(f"\nrouted candidate {i}:\n  summary   : {c['summary']}")
    for h in c["highlights"]:
        print(f"  highlight : {h}")


## 17. Assembly at the delivery boundary

The output parts now come from separate calls, potentially using different models and instruction sections. Before delivery, the harness combines them and validates the complete document.

This happens in three steps:

1. **Evaluate and repair each part.** The process from Section 13 applies to the separately generated candidates.
2. **Assemble and validate the document.** The harness combines the selected parts with known facts, then runs `validate_assembly` from Section 14 on the complete delivery object.
3. **Record provenance.** Each part is linked to its generating model, compiled instruction digest, skill version, validation status, and score.

These records support later investigation. If a subject matter expert identifies an incorrect bullet, we can trace it to the model and instructions used to generate that part.

The `ready` flag indicates whether the document passed the required checks. If a part fails or assembly validation finds a problem, the harness returns the document with diagnostics and marks it as undeliverable.

In [ ]:
@dataclass
class Delivery:
    document: Dict[str, object]
    provenance: Dict[str, Dict[str, object]]
    problems: List[str]
    ready: bool


def deliver(pipeline_result, *, routes, tone_=TONE, audience=AUD, facts=None) -> Delivery:
    document = dict(pipeline_result["document"])
    problems = validate_assembly(document, tone_, audience, facts)
    provenance = {}
    for o in pipeline_result["outcomes"]:
        part = next(p for p in OUTPUT_SECTIONS if p.name == o.part)
        route = (routes or {}).get(o.part, {"model": GENERATOR})
        provenance[o.part] = {
            "model": route["model"],
            "surface_digest": surface_digest(generation_surface(part, tone_, audience)),
            "skill": f"{tone_}-{audience}",
            "status": o.status, "attempts": o.attempts,
            "score": round(o.score, 3), "agreement": round(o.agreement, 3),
            "gate_rejected": o.gate_rejected, "review": o.review,
        }
    ready = pipeline_result["deliverable"] and not problems
    return Delivery(document, provenance, problems, ready)


def render_markdown(d: Delivery) -> str:
    """What the frontend receives. The same shape the export notebook rebuilds
    from a trace, which is why it is worth having one renderer, here."""
    lines = [f"**Summary.** {d.document['summary']}", ""]
    lines += [f"- {h}" for h in d.document["highlights"]]
    if d.document.get("breaking"):
        lines += ["", "> This release contains a breaking change."]
    return "\n".join(lines)


t0 = time.perf_counter()
split_result = await judge_and_assemble(ROUTED_CANDIDATES, TONE, AUD, panel=PANEL,
                                        facts=RELEASE_FACTS, user_input=RELEASE_INPUT)
WALL["judge + repair (split)"] = time.perf_counter() - t0

DELIVERY = deliver(split_result, routes=PART_ROUTES, facts=RELEASE_FACTS)
print(f"ready: {DELIVERY.ready}   problems: {DELIVERY.problems or 'none'}\n")
print(f"{'part':<12}{'model':<26}{'status':<16}{'score':>6}{'agree':>7}  surface")
for part, p in DELIVERY.provenance.items():
    print(f"{part:<12}{p['model']:<26}{p['status']:<16}{p['score']:>6.2f}"
          f"{p['agreement']:>7.2f}  {p['surface_digest']}")

print("\n--- what the frontend receives ---")
print(render_markdown(DELIVERY))


In [ ]:
# The boundary has to hold when a part did not make it. Here the highlights part
# is marked failed, as the repair loop would mark it after exhausting attempts.
failed = {**split_result,
          "outcomes": [o if o.part != "highlights"
                       else PartOutcome(o.part, o.chosen, o.score, o.agreement,
                                        STATUS_FAILED, "no candidate passed", 1)
                       for o in split_result["outcomes"]],
          "deliverable": False}
blocked = deliver(failed, routes=PART_ROUTES, facts=RELEASE_FACTS)

print(f"ready: {blocked.ready}")
for part, p in blocked.provenance.items():
    print(f"   {part:<12}{p['status']:<18}{p.get('attempts')} attempt(s)")
print("\nThe document still exists and is still returned — a person can look at it.")
print("What it does not do is pass a boundary that says it was accepted.")


## 18. Comparing whole-document and split generation

The comparison measures four aspects of execution. Splitting generation affects each differently, so a smaller prompt per call doesn’t necessarily mean a cheaper run overall.

**Largest individual prompt.** Whole-document generation receives the complete compiled instructions. Each split call receives only the sections relevant to its part, reducing the instruction context for that call.

**Total prompt tokens.** Shared requirements, such as prohibitions and typography rules, appear in multiple split calls. This duplication can increase total prompt tokens even when each individual prompt is smaller. The overall cost also depends on model choice, output tokens, and retries.

**Wall-clock time.** Both paths already generate candidates concurrently, so any measured improvement can’t be attributed simply to adding parallelism. Split calls produce less output individually, which can shorten their duration. Running them concurrently allows those durations to overlap, although total elapsed time still depends on the slowest call, provider latency, and concurrency limits.

**Repair scope.** Targeted repair regenerates only the failing part and retains accepted content elsewhere. Compared with regenerating the entire document, this avoids replacing unaffected output on each retry. Our repair loop already supports this for whole-document candidates; splitting initial generation extends that independence to candidate creation rather than introducing it at repair time.

In [ ]:
def usage_where(prefix: str):
    return [u for u in USAGE if u["label"].startswith(prefix)]


whole_calls = usage_where("generate[")
split_calls = usage_where("split:")

def block(name, calls, wall):
    """One row. `wall` is wall clock: for calls that fanned out, the longest
    call, not their sum — the whole-document candidates were generated with the
    same `asyncio.gather` the split uses, and comparing a fan-out against a
    serial sum would flatter the split for free."""
    p = [u["prompt_tokens"] for u in calls] or [0]
    return {"name": name, "calls": len(calls), "largest_prompt": max(p),
            "total_prompt": sum(p),
            "total_completion": sum(u["completion_tokens"] for u in calls),
            "sum_of_calls": sum(u["seconds"] for u in calls), "wall": wall}


rows = [block("whole document", whole_calls,
              max((u["seconds"] for u in whole_calls), default=0.0)),
        block("split (one model)", split_calls, WALL["split generation"])]

print(f"{'generation':<20}{'calls':>6}{'largest prompt':>16}{'total prompt':>14}"
      f"{'sum of calls':>14}{'wall clock':>12}")
for r in rows:
    print(f"{r['name']:<20}{r['calls']:>6}{r['largest_prompt']:>16}{r['total_prompt']:>14}"
          f"{r['sum_of_calls']:>13.1f}s{r['wall']:>11.1f}s")

a, b = rows[0], rows[1]
print(f"\nlargest single prompt : {a['largest_prompt']} -> {b['largest_prompt']} "
      f"({b['largest_prompt'] / a['largest_prompt'] - 1:+.0%})")
print(f"total prompt tokens   : {a['total_prompt']} -> {b['total_prompt']} "
      f"({b['total_prompt'] / a['total_prompt'] - 1:+.0%})")
print(f"wall clock            : {a['wall']:.1f}s -> {b['wall']:.1f}s")

# Blast radius: what one rejected part costs to redo, either way. On a skill
# this small the prompt sizes are close — most of a part's surface is the
# sections that govern every part. The output side is where it separates, and
# it separates by part count, not by skill size.
per_part = {p.name: ntok_(generation_surface(p, TONE, AUD)) for p in OUTPUT_SECTIONS}
n_parts = len(OUTPUT_SECTIONS)
print(f"\none part is rejected. Redoing it costs:")
print(f"   whole document   : {whole:>4} prompt tokens, {n_parts} of {n_parts} parts rewritten")
for name, t in per_part.items():
    print(f"   split/{name:<10}: {t:>4} prompt tokens, 1 of {n_parts} parts rewritten")
print(f"\n   accepted output discarded by a whole-document repair: "
      f"{(n_parts - 1) / n_parts:.0%} of the parts"
      f"  ->  {'(n-1)/n':>8}, so it grows with every part you add")


## 19. Recording the run

The measurements collected so far remain in memory unless we save them. Recording the run lets us compare configurations, share results with a subject matter expert, and use observed failures as regression cases.

The trace structure supports the analysis in the next notebook:

- **One trace per document** groups its generation, evaluation, and repair activity.
- **One observation per model call** records the model, token usage, latency, and associated part, allowing cost and timing to be analysed by part.
- **The compiled instruction digest and skill variant** identify the instructions used for the call.
- **A session ID per experiment** groups related document traces for comparison with other configurations.

If Langfuse credentials aren’t configured, the cell reports that recording was skipped. The notebook remains runnable, and the export notebook includes a saved export for readers who skip this step.

In [ ]:
RUN_ID = f"ch04-{int(time.time())}"          # the session: one experiment
EXPERIMENT = "split+routed"                   # what varied, for the comparison


def _langfuse():
    try:
        from langfuse import get_client
    except ModuleNotFoundError:
        return None
    if not (os.getenv("LANGFUSE_PUBLIC_KEY") and os.getenv("LANGFUSE_SECRET_KEY")):
        return None
    try:
        client = get_client()
        return client if client.auth_check() else None
    except Exception as exc:
        print(f"langfuse unavailable ({type(exc).__name__}); skipping the recording")
        return None


def record_run(delivery: Delivery, *, calls, run_id=RUN_ID, experiment=EXPERIMENT):
    """One trace: the document, and one observation per model call.

    Never raises. Recording a run must not be able to fail the run — the first
    rule of instrumentation, and the one people most often break.
    """
    client = _langfuse()
    if client is None:
        print("no Langfuse credentials — nothing recorded (the pipeline is unaffected)")
        return None
    try:
        with client.start_as_current_observation(
                as_type="span", name="release_notes.run",
                input={"release_input": RELEASE_INPUT, "skill": f"{TONE}-{AUD}"},
                metadata={"experiment": experiment,
                          "routes": {k: v["model"] for k, v in PART_ROUTES.items()},
                          "provenance": delivery.provenance}) as root:
            client.update_current_trace(
                session_id=run_id, user_id="ch04-notebook",
                tags=[f"experiment:{experiment}", f"skill:{TONE}-{AUD}",
                      *{f"model:{v['model']}" for v in PART_ROUTES.values()}])
            # One span per part, carrying the text that won and where it came
            # from. The generations below say what each call cost; these say
            # what the document is made of — which is what lets the export
            # notebook rebuild it without trusting a single summary field.
            for part, prov in delivery.provenance.items():
                with client.start_as_current_observation(
                        as_type="span", name=f"part.{part}",
                        input={"surface_digest": prov["surface_digest"]},
                        metadata=prov) as span:
                    span.update(output={"text": delivery.document.get(part)})
            for u in calls:
                part = u["label"].split(":")[-1].split("[")[0]
                with client.start_as_current_observation(
                        as_type="generation", name=u["label"], model=u["model"],
                        metadata={"part": part, "seconds": round(u["seconds"], 2)},
                        usage_details={"input": u["prompt_tokens"],
                                       "output": u["completion_tokens"]}):
                    pass
            root.update(output={"document": delivery.document,
                                "ready": delivery.ready,
                                "markdown": render_markdown(delivery)})
        client.flush()
        print(f"recorded {len(calls)} call(s) under session `{run_id}`")
        return run_id
    except Exception as exc:
        print(f"recording failed ({type(exc).__name__}: {str(exc)[:120]}) — "
              f"the run itself is unaffected")
        return None


RECORDED = record_run(DELIVERY, calls=usage_where("split:") + usage_where("judge:")
                      + usage_where("repair:"))
print(f"\nsession id for the export notebook: {RUN_ID}")


## 20. Testing the loop without models

The loop accepts judge callables as parameters, allowing us to replace live judges with stubs in tests. The same control flow can then run in CI without API credentials, inference costs, or variation in model responses.

Controlled responses let us test specific execution paths. For example, a judge that consistently rejects a part lets us verify that the loop stops at the repair limit and marks the document as undeliverable.

These tests check the harness logic. Evaluation with real models is still needed to assess the generated content and the judges’ behavior.

In [ ]:
def stub_verdicts(spec):
    return [JudgeVerdict(i, s, v, e, ax) for i, (s, v, e, ax) in enumerate(spec)]


def _ok(i):
    return (0.8 - 0.05 * i, VERDICT_OK, "compliant",
            {"rule_compliance": 0.9, "specificity": 0.8, "readability": 0.9})


def _bad(i):
    return (0.4 - 0.05 * i, VERDICT_VIOLATION, "reads like marketing, not a release note",
            {"rule_compliance": 0.2, "specificity": 0.5, "readability": 0.6})


async def rejects_summary_only(system, user, candidates):
    bad = "`summary`" in system
    return stub_verdicts([(_bad if bad else _ok)(i) for i, _ in enumerate(candidates)])


async def accepts_everything(system, user, candidates):
    return stub_verdicts([_ok(i) for i, _ in enumerate(candidates)])


async def stubborn_rewrite(part, reasons, rules, current):
    return current            # the repair changes nothing

mixed_panel = {"a": rejects_summary_only, "b": rejects_summary_only,
               "c": accepts_everything}
failed = await judge_and_assemble(CANDIDATES, TONE, AUD, panel=mixed_panel,
                                  rewrite=stubborn_rewrite, facts=RELEASE_FACTS)

for o in failed["outcomes"]:
    print(f"{o.part:12} status={o.status:18} attempts={o.attempts}  {o.detail[:80]}")
print(f"\ndeliverable: {failed['deliverable']}")
print("schema problems:", validate_assembly(failed["document"], TONE, AUD, RELEASE_FACTS) or "none")

by_part = {o.part: o for o in failed["outcomes"]}
assert failed["deliverable"] is False                      # one bad part poisons the run
assert by_part["summary"].status == STATUS_FAILED          # unfixable -> terminal
assert by_part["highlights"].status == STATUS_OK           # untouched by its neighbour

# The gate is authoritative even when a panel would have passed the text.
over_limit = [{"summary": " ".join(["word"] * (word_limit(TONE, AUD) + 3)),
               "highlights": CANDIDATES[0]["highlights"]}]
gate = deterministic_problems("summary", over_limit[0]["summary"], TONE, AUD)
assert gate and "words" in gate[0]
print(f"\ngate on an over-long summary: {gate[0][:80]}")
print("\nCI assertions passed - no model, no spend, no flake")

summary      status=failed_validation  attempts=1  still rejected after 1 attempt(s): [a] reads like marketing, not a release note;
highlights   status=ok                 attempts=0  gate passed, panel majority OK

deliverable: False
schema problems: none

gate on an over-long summary: 28 words, the skill allows at most 25 -- One sentence, at most 25 words, written

CI assertions passed - no model, no spend, no flake


## 21. Measuring token usage and cost

We report two measurements that answer different questions.

**Rule tokens** compare the instruction text sent to judges in portioned and whole-document evaluation. This excludes candidate content, request scaffolding, output tokens, and caching effects, so it measures instruction overhead rather than total cost.

In this example, portioned judging uses more rule tokens per evaluation round because shared sections are repeated for each part. Targeted repair can reduce subsequent usage by evaluating only the affected part. The overall difference depends on how much content and instruction text each call needs.

**Model cost** is calculated from measured token usage and OpenRouter’s published per-token rates.

In the recorded run, judging costs substantially more than generation. Each part is evaluated by three judges, adding repeated rubric and candidate input alongside the judges’ responses. That cost provides multiple assessments and a measure of their agreement. The per-part configuration also lets us test less expensive panels for routine content and reserve more expensive models for parts that warrant them.

Deterministic gates add no model inference cost. They handle measurable requirements before judging, avoiding model calls for checks such as word counts that code can perform directly.

In [ ]:
try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
    def ntok(t): return len(_enc.encode(t))
    counter = "tiktoken cl100k_base"
except Exception:
    def ntok(t): return max(1, round(len(t) / 4))
    counter = "approximation (chars/4)"

whole = ntok(whole_leaf(TONE, AUD))
rows = [(p.name, ntok(rubric(p, TONE, AUD))) for p in OUTPUT_SECTIONS]
n_parts, n_judges = len(rows), len(PANEL_MODELS)
portioned = sum(t for _, t in rows)

print(f"RULES TOKENS ({counter})")
print(f"  {'judged part':<14}{'rubric':>8}{'whole skill':>13}{'share':>8}")
for name, t in rows:
    print(f"  {name:<14}{t:>8}{whole:>13}{t / whole:>7.0%}")
print(f"  {'whole-document judge, whole skill':<34}: {whole:>5}  (x{n_judges} judges = {whole * n_judges})")
print(f"  {f'portioned, {n_parts} parts':<34}: {portioned:>5}  (x{n_judges} judges = {portioned * n_judges})")
print(f"  {'difference':<34}: {portioned - whole:+5}  ({(portioned - whole) / whole:+.0%})")
print(f"  {'repair: whole doc vs one part':<34}: {whole:>5} vs {max(t for _, t in rows)}")


@lru_cache(maxsize=1)
def pricing() -> Dict[str, Tuple[float, float]]:
    '''(prompt, completion) USD per token, from OpenRouter's model catalogue.'''
    req = urllib.request.Request(f"{BASE_URL}/models",
                                 headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"})
    data = json.load(urllib.request.urlopen(req, timeout=60))["data"]
    return {m["id"]: (float(m["pricing"]["prompt"]), float(m["pricing"]["completion"]))
            for m in data}


def cost_of(u) -> float:
    p_in, p_out = pricing().get(u["model"], (0.0, 0.0))
    return u["prompt_tokens"] * p_in + u["completion_tokens"] * p_out


by_label = {}
for u in USAGE:
    key = u["label"].split("[")[0]
    agg = by_label.setdefault(key, {"n": 0, "in": 0, "out": 0, "s": 0.0, "model": u["model"], "cost": 0.0})
    agg["n"] += 1; agg["in"] += u["prompt_tokens"]; agg["out"] += u["completion_tokens"]
    agg["s"] += u["seconds"]; agg["cost"] += cost_of(u)

print(f"\nACTUAL CALLS ({len(USAGE)} total)")
print(f"  {'phase':<18}{'calls':>6}{'in':>8}{'out':>7}{'sum sec':>9}{'cost':>10}")
for label, a in sorted(by_label.items()):
    print(f"  {label:<18}{a['n']:>6}{a['in']:>8}{a['out']:>7}{a['s']:>9.1f}{'$' + format(a['cost'], '.5f'):>10}")
tot_s = sum(u["seconds"] for u in USAGE)
print(f"  {'TOTAL':<18}{len(USAGE):>6}{sum(u['prompt_tokens'] for u in USAGE):>8}"
      f"{sum(u['completion_tokens'] for u in USAGE):>7}{tot_s:>9.1f}"
      f"{'$' + format(sum(cost_of(u) for u in USAGE), '.5f'):>10}")
print("\n  Priced from the published per-token rates against measured tokens; "
      "provider-side\n  accounting settles later and rounds, so the two are not "
      "the same number.")
print(f"  deterministic gate: {len(CANDIDATES) * len(OUTPUT_SECTIONS)} checks, $0.00000, "
      f"{0.0:.1f}s of model time")

RULES TOKENS (tiktoken cl100k_base)
  judged part     rubric  whole skill   share
  summary            134          238    56%
  highlights         135          238    57%
  whole-document judge, whole skill :   238  (x3 judges = 714)
  portioned, 2 parts                :   269  (x3 judges = 807)
  difference                        :   +31  (+13%)
  repair: whole doc vs one part     :   238 vs 135

ACTUAL CALLS (12 total)
  phase              calls      in    out  sum sec      cost
  generate               3    1167    325      9.5  $0.00062
  judge:anthropic        3    3942   1579     14.6  $0.01184
  judge:google           3    2595   1002      5.6  $0.00215
  judge:openai           3    1603    811      7.4  $0.00485
  TOTAL                 12    9307   3717     37.1  $0.01946

  cross-check, OpenRouter key accounting: $0.0000 (settles with a lag, so it can read low right after a run)
  deterministic gate: 6 checks, $0.00000, 0.0s of model time


### Latency is the unambiguous win

Parts fan out, and inside each part the panel fans out. The call graph stays one
model call deep whatever its width — and the width is what you were paying for
anyway.

In [ ]:
inflight = peak = 0
_orig_typed_call = typed_call


async def counting_typed_call(**kwargs):
    global inflight, peak
    inflight += 1; peak = max(peak, inflight)
    try:
        return await _orig_typed_call(**kwargs)
    finally:
        inflight -= 1


typed_call = counting_typed_call
t0 = time.perf_counter()
await asyncio.gather(*[
    score_group(PANEL, judge_system(p, len(CANDIDATES), rubric(p, TONE, AUD)), "",
                [as_text(c[p.name]) for c in CANDIDATES])
    for p in OUTPUT_SECTIONS])
elapsed = time.perf_counter() - t0
typed_call = _orig_typed_call

calls = len(OUTPUT_SECTIONS) * len(PANEL)
serial = sum(u["seconds"] for u in USAGE[-calls:])
print(f"parts {len(OUTPUT_SECTIONS)} x judges {len(PANEL)} = {calls} judge calls")
print(f"peak concurrent : {peak}")
print(f"wall clock      : {elapsed:.1f}s")
print(f"sum of calls    : {serial:.1f}s  -> {serial / elapsed:.1f}x speedup from fan-out")

parts 2 x judges 3 = 6 judge calls
peak concurrent : 6
wall clock      : 4.6s
sum of calls    : 20.8s  -> 4.5x speedup from fan-out


## 22. Adapting the pipeline to your own skill

**Add an axis value**, such as `tone: technical`, by writing a fragment for the sections that change. `discover_axes` includes the new value in the generated combinations, while `must_replace` checks that required placeholders are replaced. Check the resulting combinations against your intended variant list.

**Add an axis**, such as `locale`, by including it in `AXIS_ORDER` and defining `applies_when.locale` in the relevant fragments. Variant IDs and filenames are derived from those definitions.

**Add a field** by first deciding who supplies its value. A known fact belongs in `HARNESS_OWNED` and is injected during assembly. Generated content that needs evaluation gets an `OutputSection` and a rubric. In both cases, update the skill’s contract section and the executable schemas. The contract-key check in Section 8 detects mismatches between them.

**Add a rule** to the fragment and section responsible for it. Then decide how to check it. A requirement that can be evaluated deterministically belongs in `deterministic_problems`, with its threshold or rule data read from the skill. A requirement that needs interpretation belongs in the relevant rubric.

**Add a judge** through an entry in `PANEL_MODELS`, including its request-mode and routing settings. A model from another family can add diversity to the panel, but its suitability still needs evaluation.

### Where each question is checked

| Question | Mechanism | Additional model calls |
|---|---|---|
| Does each section have one replacement owner? | Compiler ownership checks | None |
| Which fragments contributed to an instruction section? | Manifest contributors and source hashes | None |
| Does the compiled text match the reviewed version? | Compiled content hash comparison | None |
| Are instructions for an excluded variant absent? | Static assertions on the compiled instructions | None |
| Do schema keys agree with the skill’s contract? | Contract-key check at startup | None |
| Is the summary within the word limit? | Deterministic gate using the skill’s threshold | None |
| Does the output meet bullet limits and avoid listed terms? | Deterministic gate | None |
| Does the assembled object satisfy the delivery schema? | Envelope validation | None |
| Does a persistent violation stop delivery? | CI tests with injected judges | None |
| Is the wording appropriate for a developer audience? | Judge panel using the relevant rubric | One call per judge |
| Where do the judges disagree? | Per-axis standard deviation | None beyond the panel calls |
| Did the repair resolve the problem? | Deterministic checks followed by comparative judging | A further evaluation round |

The compiler handles assembly constraints, and deterministic checks handle measurable requirements. Models generate the content and evaluate requirements that need interpretation.

### Keeping rule definitions in the skill

> Maintain rule text, thresholds, and permitted values in the skill. The harness implements the mechanisms that read, apply, and check them.

If a checker paraphrases an instruction or hard-codes a threshold already defined in the skill, both copies need to be maintained. Reading those definitions from the compiled artifact keeps the checks aligned with the instructions supplied to the model.